# 任意分辨率视觉

真实的图像不是224x224的方块。一张收据是9:16，一张图票是16:9，一份医疗扫描可能是4096x4096，一张手机截图是9:19.5。2024年以前的答案是，将图像缩放到固定的方块，但是会使得OCR、文档理解以及高分辨率解析任务失败。NaViT 表明可以将不同分辨率的patches打包进单独transformer的批，使用块对焦掩码。Qwen2 的M-RoPE完全丢掉了绝对位置表。LLaVA的AnyRes 将高分辨图片变成子图和全局缩略图。SigLIP的NaFlex变体如今是那些想用单个checkpoint服务所有长宽比的开源VLM的默认解码器，

## 问题描述

Transformer 需要一个序列，一个批是相同长度序列的堆叠。如果你的图像是224x224，你每次都能得到196个Patch Token，不需要对齐，完美～在224上训练，在224上推理，永远不要考虑分辨率的问题。

但是世界不配合，图片并不是224的，2024年以前三个途径失败的原因：
- 缩放到固定方块（224x224 或 336x336），挤压会扭曲文字和人脸，降采样会毁掉图表标签和OCR内容。是LLaVA1.5以前的默认做法。
- 裁剪到固定长宽比。丢掉大部分的图像，使用裁剪区域来处理视觉问题。
- 按长轴进行填充。 修好了扭曲但是有50%的Token被浪费在填充上，需要付平方级的注意力成本。

2024-2025年的答案：让Transformer在图像的原始分辨率上吃Patches，然后找到将这些不同Patch塞到同一个序列的方案，同时不浪费计算。

## 基本概念

### NaViT 和 patch-n‘-pack

NaViT 的机制：
1. 对于批中的每张图片，按照选好的patch大小计算原始的patch网格。
2. 将每张图片的patch展平，形成一个变长序列。
3. 将所有的图片patch连接成一个长序列。
4. 构建一个块对角注意力掩码，让图A的patch只关注图A。
5. 携带每个patch的位置信息（2D RoPE 或者分数位置嵌入）

比如有3张图片，336x336（576 tokens），224x224（256 tokens），448x336（768 tokens）变成一个576+256+768 共1600 token长度的序列，然后使用1600x1600的块对角掩码。没有填充，没有注意力浪费，transformer处理任意的长宽比。

NaViT 在训练中使用了按比例丢掉patch————在批中以50%的比例丢掉patch，一方面做了正则化，一方面加速了训练。SigLIP2 继承了这一做法。

### AnyRes

LLaVA-NeXT 的AnyRes 是另一种务实的做法。给一个高分辨率的图片，以及一个固定的编码器（比如CLIP 或者 SigCLIP 336），将图片切块：
1. 从预定义集合中选者一个最适合原图片长宽比的网格布局，1x1，2x2，1x3，3x1等等。
2. 将整张图片切块，每一块都是336x336的切片。
3. 同时引入缩略图，将整张图片缩放到336x336作为全局上下文Token。
4. 将每个块送进冻结的编码器。将块的Token和缩略图的Token进行连接。

对于一张672x672的图片，产生2x2个块，以及一个缩略图，所以是576x4+576共2880个视觉token。贵但有效，LLM同时能看见局部细节和全局上下文。

AnyRes 是在你的编码器冻结且只支持一种分辨率场景下的选择。不过注意，对于大分辨率，视觉token数量会爆炸。

### M-RoPE

Qwen2-VL 引入了多模态旋转位置编码。与NaViT的分数位置和AnyRes的瓦片图+缩略图组合不一样，每个patch都带着三维位置信息（时间、高度、宽度），Q/K旋转可以处理任意的H、W和时间跨度。

M-RoPE 在不重训的情况下可以天然处理动态分辨率。推理的时候你给了一张HxW的图片，编码器产生H/14xW/14个token，每个token都有自己的（t=0，r=row，c=col）位置信息，RoPE用正确的频率旋转注意力。

不像AnyRes，M-RoPE在原始分辨率上生成HxW/P^2个token，不再需要乘法倍增的瓦片。也不像NaViT，一次前向过程中AnyRes只处理一张图片，不需要处理交叉注意力。

### NaFlex

NaFlex 是SigLIP2 checkpoint 的原生灵活模式。一个模型可以在推理阶段处理多种不同的序列长度（256，729，1024 tokens）。内部实际上在训练时使用NaViT风格的patch-n‘-pack以及绝对分数位置编码。买点在于，有一个检查点，可以根据下游推理任务选择你的token预算。

对于语义相关任务（分类，检索）使用256个token，对于OCT或者图表理解，1024个token，不需要重训。


# 开始编码

教学积木：任意分辨率三条路——**NaViT patch-n-pack（块对角 mask）**、**AnyRes（瓦片+缩略图）**、**M-RoPE（t/h/w 旋转位置）**。


## 1. NaViT：变长 patch 打包 + 块对角注意力掩码


In [1]:
from __future__ import annotations

import math
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class TinyAnyResConfig:
    """任意分辨率教学配置（尺寸故意偏小）。"""

    patch_size: int = 16
    """每个 patch 边长（像素）。"""

    vit_dim: int = 32
    """patch 嵌入维。"""

    n_heads: int = 4
    encoder_size: int = 64
    """AnyRes 固定编码器输入边长（对应真模型 336）。"""

    rope_dim: int = 16
    """RoPE 作用在每个 head 上的旋转维（需为偶数且 <= head_dim）。"""


def image_to_patches(
    image: torch.Tensor,
    patch_size: int,
) -> tuple[torch.Tensor, int, int]:
    """
    将单张图像切成非重叠 patch 并展平。

    Args:
        image: ``(3, H, W)``，要求 ``H % patch_size == 0`` 且 ``W % patch_size == 0``。
        patch_size: patch 边长。

    Returns:
        patches: ``(Gh * Gw, 3 * patch_size * patch_size)`` 展平像素。
        grid_h: 行方向 patch 数 ``H // patch_size``。
        grid_w: 列方向 patch 数 ``W // patch_size``。
    """
    if image.ndim != 3:
        raise ValueError(f"expect (3,H,W), got {tuple(image.shape)}")
    _, H, W = image.shape
    if H % patch_size or W % patch_size:
        raise ValueError(f"H,W must divide patch_size={patch_size}, got {(H, W)}")
    gh, gw = H // patch_size, W // patch_size
    # (3, gh, ps, gw, ps) -> (gh, gw, 3, ps, ps) -> (gh*gw, 3*ps*ps)
    x = image.reshape(3, gh, patch_size, gw, patch_size)
    x = x.permute(1, 3, 0, 2, 4).reshape(gh * gw, -1)
    return x, gh, gw


class PatchEmbed(nn.Module):
    """线性把展平 patch 投到 ``vit_dim``（玩具 patch embedding）。"""

    def __init__(self, patch_size: int, vit_dim: int) -> None:
        super().__init__()
        self.proj = nn.Linear(3 * patch_size * patch_size, vit_dim)

    def forward(self, flat_patches: torch.Tensor) -> torch.Tensor:
        """
        Args:
            flat_patches: ``(N, 3*P*P)``。

        Returns:
            tokens: ``(N, vit_dim)``。
        """
        return self.proj(flat_patches)


def pack_variable_images(
    images: list[torch.Tensor],
    patch_size: int,
    embed: PatchEmbed,
) -> tuple[torch.Tensor, list[int], list[tuple[int, int]]]:
    """
    NaViT 风格 patch-n-pack：多张不同分辨率图 -> 一条长 token 序列。

    Args:
        images: 长度 B 的列表，每个 ``(3, H_i, W_i)``。
        patch_size: 共用 patch 边长。
        embed: patch 嵌入层。

    Returns:
        packed: ``(1, sum N_i, vit_dim)`` 打包后的 batch=1 序列。
        lengths: 每张图的 token 数 ``[N_0, N_1, ...]``。
        grids: 每张图 ``(grid_h, grid_w)``。
    """
    tokens_list: list[torch.Tensor] = []
    lengths: list[int] = []
    grids: list[tuple[int, int]] = []
    for img in images:
        flat, gh, gw = image_to_patches(img, patch_size)
        tok = embed(flat)
        tokens_list.append(tok)
        lengths.append(tok.size(0))
        grids.append((gh, gw))
    packed = torch.cat(tokens_list, dim=0).unsqueeze(0)  # (1, sumN, D)
    return packed, lengths, grids


def build_block_diagonal_mask(
    lengths: list[int],
    device: torch.device | str,
) -> torch.Tensor:
    """
    块对角注意力掩码：图 i 的 patch 只能看见图 i。

    Args:
        lengths: 每张图 token 长度。
        device: 掩码所在设备。

    Returns:
        attn_mask: ``(L, L)``，可见为 ``0``，屏蔽为 ``-inf``（供 ``MultiheadAttention``）。
    """
    L = sum(lengths)
    mask = torch.full((L, L), float("-inf"), device=device)
    start = 0
    for n in lengths:
        mask[start : start + n, start : start + n] = 0.0
        start += n
    return mask


print("NaViT pack helpers ready")


NaViT pack helpers ready


## 2. AnyRes：选网格 → 瓦片 + 全局缩略图 → 拼接视觉 token


In [2]:
# 预定义候选网格 (gh, gw)：块数 = gh*gw（另加 1 张缩略图）
ANYRES_GRIDS: list[tuple[int, int]] = [
    (1, 1),
    (1, 2),
    (2, 1),
    (2, 2),
    (1, 3),
    (3, 1),
]


def select_anyres_grid(
    height: int,
    width: int,
    candidates: list[tuple[int, int]] = ANYRES_GRIDS,
) -> tuple[int, int]:
    """
    选与原图长宽比最接近的瓦片网格。

    Args:
        height: 原图高。
        width: 原图宽。
        candidates: 候选 ``(grid_h, grid_w)``。

    Returns:
        best: 最优 ``(grid_h, grid_w)``。
    """
    aspect = width / max(height, 1)
    best = candidates[0]
    best_score = float("inf")
    for gh, gw in candidates:
        score = abs((gw / gh) - aspect)
        # 同分时偏好更少的块，省 token
        score = score + 1e-3 * (gh * gw)
        if score < best_score:
            best_score = score
            best = (gh, gw)
    return best


def resize_image(image: torch.Tensor, size_h: int, size_w: int) -> torch.Tensor:
    """
    Args:
        image: ``(3, H, W)``。
        size_h: 目标高。
        size_w: 目标宽。

    Returns:
        resized: ``(3, size_h, size_w)``。
    """
    x = image.unsqueeze(0)
    x = F.interpolate(x, size=(size_h, size_w), mode="bilinear", align_corners=False)
    return x.squeeze(0)


class FixedSizeEncoder(nn.Module):
    """
    冻结固定分辨率编码器示意（对应 CLIP/SigLIP 只吃 encoder_size×encoder_size）。

    调用方负责先把图缩放到 ``encoder_size`` 方图；本模块只做切 patch + 嵌入。
    输出固定 ``(S*S, vit_dim)``，S = encoder_size // patch_size。
    """

    def __init__(self, cfg: TinyAnyResConfig) -> None:
        super().__init__()
        self.cfg = cfg
        self.side = cfg.encoder_size // cfg.patch_size
        self.embed = PatchEmbed(cfg.patch_size, cfg.vit_dim)
        for p in self.parameters():
            p.requires_grad = False

    def forward(self, image: torch.Tensor) -> torch.Tensor:
        """
        Args:
            image: ``(3, encoder_size, encoder_size)``，**调用方先缩放好**的方图。

        Returns:
            tokens: ``(S*S, vit_dim)``。
        """
        _, H, W = image.shape
        S = self.cfg.encoder_size
        if (H, W) != (S, S):
            raise ValueError(
                f"FixedSizeEncoder expects {(S, S)}, got {(H, W)}; "
                "请先 resize_image 再 encode（缩略图/瓦片都如此）"
            )
        flat, _, _ = image_to_patches(image, self.cfg.patch_size)
        return self.embed(flat)


def anyres_encode(
    image: torch.Tensor,
    encoder: FixedSizeEncoder,
    candidates: list[tuple[int, int]] = ANYRES_GRIDS,
) -> tuple[torch.Tensor, tuple[int, int], int]:
    """
    LLaVA-NeXT AnyRes：缩略图 + 多瓦片，各自过固定编码器后拼接。

    Args:
        image: ``(3, H, W)`` 任意分辨率（不必能被 patch 整除；先缩放到网格画布）。
        encoder: 固定边长编码器（只接受 ``SxS`` 输入）。
        candidates: AnyRes 网格候选。

    Returns:
        tokens: ``((1 + gh*gw) * S*S, vit_dim)``，顺序为 ``[缩略图 | tile_0 | ...]``。
        grid: 选用的 ``(gh, gw)``。
        tokens_per_tile: 每个瓦片/缩略图的 token 数 ``S*S``。
    """
    _, H, W = image.shape
    gh, gw = select_anyres_grid(H, W, candidates)
    S = encoder.cfg.encoder_size

    # 缩略图：先缩放到 SxS，再 encode（全局上下文）
    thumb_img = resize_image(image, S, S)
    thumb = encoder(thumb_img)

    # 画布：缩放到 (gh*S, gw*S) 再切 gh*gw 个已是 SxS 的瓦片，再 encode
    canvas = resize_image(image, gh * S, gw * S)
    tiles: list[torch.Tensor] = []
    for i in range(gh):
        for j in range(gw):
            tile = canvas[:, i * S : (i + 1) * S, j * S : (j + 1) * S]
            tiles.append(encoder(tile))
    tokens = torch.cat([thumb, *tiles], dim=0)
    return tokens, (gh, gw), thumb.size(0)


print("AnyRes helpers ready")


AnyRes helpers ready


## 3. M-RoPE：给每个 patch 三维坐标 (t, h, w) 并旋转 Q/K


In [ ]:
def build_thw_position_ids(
    grid_h: int,
    grid_w: int,
    time_id: int = 0,
    device: torch.device | str = "cpu",
) -> torch.Tensor:
    """
    为单帧图像的 patch 网格生成 ``(t, h, w)`` 位置。

    Args:
        grid_h: patch 行数。
        grid_w: patch 列数。
        time_id: 时间维（单图为 0；视频第 T 帧为 T）。
        device: 设备。

    Returns:
        pos: ``(grid_h * grid_w, 3)``，每行为 ``(t, row, col)``。
    """
    rows = []
    for r in range(grid_h):
        for c in range(grid_w):
            rows.append((time_id, r, c))
    return torch.tensor(rows, device=device, dtype=torch.long)


def rotate_half(x: torch.Tensor) -> torch.Tensor:
    """
    RoPE 辅助：将最后一维两两配对后交换并取负。

    Args:
        x: ``(..., rope_dim)``，``rope_dim`` 为偶数。

    Returns:
        y: 与 ``x`` 同形。
    """
    x1 = x[..., ::2]
    x2 = x[..., 1::2]
    return torch.stack((-x2, x1), dim=-1).flatten(-2)


def apply_mrope(
    q: torch.Tensor,
    k: torch.Tensor,
    pos_thw: torch.Tensor,
    rope_dim: int,
    base: float = 10000.0,
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    简化版 M-RoPE：把 ``rope_dim`` 拆成三段，分别用 t/h/w 做旋转。

    真 Qwen2-VL 的频率分配更细；这里只演示「多轴位置 -> 旋 Q/K」。

    Args:
        q: ``(B, H, L, head_dim)``。
        k: ``(B, H, L, head_dim)``。
        pos_thw: ``(L, 3)`` 每个 token 的 ``(t, h, w)``。
        rope_dim: 参与旋转的末维长度（偶数，且 ``<= head_dim``，并最好能被 3 整除）。
        base: RoPE 基数。

    Returns:
        q_rot, k_rot: 与输入同形；仅最后 ``rope_dim`` 维被旋转，其余维原样。
    """
    if rope_dim % 2:
        raise ValueError("rope_dim must be even")
    # 三轴尽量均分；余数给 w
    d = rope_dim // 2  # 复数对数
    split = [d // 3, d // 3, d - 2 * (d // 3)]
    # 对 q/k 的 rope 部分按轴旋转
    def _axis_angles(pos: torch.Tensor, n_complex: int) -> torch.Tensor:
        # pos: (L,) -> angles (L, n_complex)
        if n_complex == 0:
            return pos.new_zeros(pos.size(0), 0)
        idx = torch.arange(n_complex, device=pos.device, dtype=torch.float32)
        inv_freq = 1.0 / (base ** (idx / max(n_complex, 1)))
        return pos.float().unsqueeze(1) * inv_freq.unsqueeze(0)

    t, h, w = pos_thw[:, 0], pos_thw[:, 1], pos_thw[:, 2]
    ang = torch.cat(
        [_axis_angles(t, split[0]), _axis_angles(h, split[1]), _axis_angles(w, split[2])],
        dim=-1,
    )  # (L, d)
    cos = torch.cos(ang).to(q.dtype)
    sin = torch.sin(ang).to(q.dtype)
    # 扩到 (1,1,L,rope_dim) 以便广播
    cos = torch.stack((cos, cos), dim=-1).flatten(-2).view(1, 1, -1, rope_dim)
    sin = torch.stack((sin, sin), dim=-1).flatten(-2).view(1, 1, -1, rope_dim)

    def _rot(x: torch.Tensor) -> torch.Tensor:
        x_rot, x_pass = x[..., :rope_dim], x[..., rope_dim:]
        x_rot = x_rot * cos + rotate_half(x_rot) * sin
        return torch.cat([x_rot, x_pass], dim=-1)

    return _rot(q), _rot(k)


print("M-RoPE helpers ready")


## 4. 拼装演示模块 + 冒烟测试（pack / AnyRes / M-RoPE）


In [ ]:
class PackedSelfAttention(nn.Module):
    """对 NaViT 打包序列做自注意力（带块对角 mask）。"""

    def __init__(self, dim: int, n_heads: int) -> None:
        super().__init__()
        self.attn = nn.MultiheadAttention(dim, n_heads, batch_first=True)
        self.norm = nn.LayerNorm(dim)

    def forward(self, x: torch.Tensor, attn_mask: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: ``(1, L, D)`` 打包序列。
            attn_mask: ``(L, L)`` 块对角 float mask。

        Returns:
            y: ``(1, L, D)``。
        """
        h, _ = self.attn(x, x, x, attn_mask=attn_mask, need_weights=False)
        return self.norm(x + h)


def smoke_test() -> None:
    """验证三条任意分辨率路径的 shape 与掩码语义。"""
    torch.manual_seed(0)
    cfg = TinyAnyResConfig()
    device = "cpu"
    embed = PatchEmbed(cfg.patch_size, cfg.vit_dim)

    # --- NaViT pack ---
    imgs = [
        torch.randn(3, 32, 32),   # 2x2 patches
        torch.randn(3, 32, 48),   # 2x3
        torch.randn(3, 48, 32),   # 3x2
    ]
    packed, lengths, grids = pack_variable_images(imgs, cfg.patch_size, embed)
    mask = build_block_diagonal_mask(lengths, device)
    attn = PackedSelfAttention(cfg.vit_dim, cfg.n_heads)
    out = attn(packed, mask)
    print("=== NaViT patch-n-pack ===")
    print(f"grids={grids}, lengths={lengths}, sum={sum(lengths)}")
    print(f"packed={tuple(packed.shape)}, out={tuple(out.shape)}")
    print(f"mask diag-block visible check: block0 finite count={torch.isfinite(mask[:lengths[0], :lengths[0]]).sum().item()}")
    # 跨图应被屏蔽
    assert not torch.isfinite(mask[0, lengths[0]]).item()

    # --- AnyRes ---
    encoder = FixedSizeEncoder(cfg)
    tall = torch.randn(3, 80, 40)  # 偏竖图 -> 倾向 2x1 / 3x1
    tokens, grid, npt = anyres_encode(tall, encoder)
    print("\n=== AnyRes ===")
    print(f"grid={grid}, tokens_per_tile={npt}, total_tokens={tokens.size(0)}")
    print(f"expect (1+gh*gw)*npt={(1 + grid[0] * grid[1]) * npt}")
    assert tokens.size(0) == (1 + grid[0] * grid[1]) * npt

    wide = torch.randn(3, 40, 80)
    _, grid_w, _ = anyres_encode(wide, encoder)
    print(f"wide image selected grid={grid_w}")

    # --- M-RoPE ---
    print("\n=== M-RoPE ===")
    gh, gw = 2, 3
    L = gh * gw
    H = cfg.n_heads
    head_dim = cfg.vit_dim // H
    # 手工构造假 Q/K
    q = torch.randn(1, H, L, head_dim)
    k = torch.randn(1, H, L, head_dim)
    pos = build_thw_position_ids(gh, gw, time_id=0, device=device)
    rope_dim = min(cfg.rope_dim, head_dim - (head_dim % 2))
    # 保证 rope_dim 可被使用
    rope_dim = rope_dim - (rope_dim % 2)
    q2, k2 = apply_mrope(q, k, pos, rope_dim=rope_dim)
    print(f"pos_thw shape={tuple(pos.shape)}, rope_dim={rope_dim}")
    print(f"q/k rotated shapes={tuple(q2.shape)}, {tuple(k2.shape)}")
    assert q2.shape == q.shape and k2.shape == k.shape
    # 改 col 应改变旋转结果
    pos2 = pos.clone()
    pos2[:, 2] += 1
    q3, _ = apply_mrope(q, k, pos2, rope_dim=rope_dim)
    assert not torch.allclose(q2, q3)
    print("SMOKE TEST OK")


smoke_test()
